# Hardware-Aware Gateway. Colab bootstrap

Runs the CUDA half of the project on a free T4.

**Set the runtime first:** Runtime > Change runtime type > T4 GPU. Cell 1
fails deliberately if you forget, because every GPU test would otherwise skip
and the run would look like it passed.


In [ ]:
import torch
assert torch.cuda.is_available(), (
    'No CUDA device. Runtime > Change runtime type > T4 GPU, then rerun.'
)
print(torch.cuda.get_device_name(0))
print('torch     ', torch.__version__)
import triton; print('triton    ', triton.__version__)


## 1. Install the package

`pip install -e .` rather than appending to `sys.path`. Every later cell shells
out with `!python -m hag...`, and a `sys.path` entry set inside this process
does not exist in that subprocess. Installing the package is what makes the
module importable from anywhere.

The transformers pin is deliberate: Colab preinstalls a 4.x, which already
satisfies `>=4.44`, so pip would decline to upgrade and leave you on an API
the code no longer calls.


In [ ]:
!git clone -q https://github.com/Rahu378/hardware-aware-gateway.git
%cd hardware-aware-gateway
!pip install -q -e '.[e2e,dev]'

# Prove it imports from a *subprocess*, which is what the later cells use.
!python -c "import hag; print('hag importable:', hag.__file__)"


## 2. Correctness before speed

The CUDA half of this suite was written on a Mac and has never executed on an
NVIDIA device. This is its first real run. Expect to fix something; that is
the point.

Do not continue past a failure. A wrong kernel still produces fast numbers.


In [ ]:
!python -m pytest -q


## 3. Baseline profile. Find the traffic jam

Profile *before* changing anything.

This uses PyTorch's own profiler, which needs no apt package and reports
per-kernel GPU time. Read the top of the table: if elementwise kernels
(`mul`, `add`, `silu`, the norms) outrank the GEMMs, the workload is
memory-bound and fusion is the right lever.


In [ ]:
!python -m hag.profile_torch --model Qwen/Qwen2.5-1.5B \
    --prompt-tokens 512 --new-tokens 32


### Optional: the Nsight Systems timeline

`nsys` shows the gaps *between* kernels, which the torch profiler cannot.
It is not in Colab's default apt sources, so this pulls NVIDIA's repo first.
It is a large download and entirely optional; the profile above already
answers which kernels dominate.

Errors are shown rather than swallowed. If the install fails, skip this and
move on.


In [ ]:
import shutil, subprocess

if shutil.which('nsys') is None:
    !wget -q https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb
    !dpkg -i cuda-keyring_1.1-1_all.deb > /dev/null
    !apt-get update -qq
    !apt-get install -y nsight-systems-cli

print('nsys:', shutil.which('nsys') or 'NOT AVAILABLE, skip this section')


In [ ]:
import shutil
if shutil.which('nsys'):
    !nsys profile --trace=cuda,nvtx,osrt --cuda-memory-usage=true \
        --force-overwrite=true -o profiles/baseline_t4 \
        python -m hag.bench_e2e --model Qwen/Qwen2.5-1.5B --prompt-tokens 512 --new-tokens 64
    !nsys stats --report cuda_gpu_kern_sum profiles/baseline_t4.nsys-rep | head -30
else:
    print('nsys unavailable; the torch profile above covers the same question.')


## 4. Op-level sweep

Writes `results/ops_nvidia-t4_fp16.json`. Note the launch-floor line at the
top: on a T4 it is far lower than on Apple silicon, so more of the decode
regime becomes measurable.


In [ ]:
!python -m hag.bench_ops --backend cuda --dtype fp16


## 5. End-to-end

The number that decides whether the kernel work mattered. A large op-level
speedup on an op occupying 4% of the forward pass moves this by 4%.


In [ ]:
!python -m hag.bench_e2e --model Qwen/Qwen2.5-1.5B \
    --prompt-tokens 512 --new-tokens 128


## 6. Regenerate the README tables

The T4 rows should now sit alongside the Apple M3 ones. If only M3 rows
appear, the sweep in section 4 did not write its JSON; scroll back and read
its output rather than continuing.


In [ ]:
!python -m hag.report
!sed -n '/BENCH:BEGIN/,/BENCH:END/p' README.md


## 7. Save the artifacts

Check the file list before downloading. An empty `results/` means something
above failed quietly.


In [ ]:
!ls -la results profiles

from google.colab import files
!zip -qr artifacts.zip results profiles
files.download('artifacts.zip')


---

### Next: Nsight Compute counters

`ncu` will fail on Colab with `ERR_NVGPUCTRPERM`. The runtime does not grant
performance-counter access. That is the one step needing a GCP or Azure VM on
signup credits. See `scripts/profile_ncu.sh` for the module-parameter fix.
